# Mean-difference probe (regular)

The reference activation probe, the simple way: **load + tokenize locally**, then
run **one remote NDIF trace per batch** to read layer-`L` activations, pool the
response span, and project onto the trained direction. Easy to follow; one queue
wait per batch.

For the efficient form — the whole dataset in **one** NDIF session, with an
early-stop after layer `L` — see [`mean_probe_fast.ipynb`](mean_probe_fast.ipynb).
Both apply the *same* probe and write the *same* `index,deceptive,score` contract.

To use either: copy the notebook **and** `mean_probe.npz` into `submission/`.

In [ ]:
import os, sys
import numpy as np
import torch

sys.path.insert(0, "submission")          # util.py lives in submission/
from util import build_model, decoder_layers, load_examples, response_span, write_submission

# The runner sets DATASET_NAME; read it, don't hard-code.
DATASET_NAME = os.environ["DATASET_NAME"]
# Optional `--limit N` -> $ALETHEIA_LIMIT: score only the first N rows (fast rehearsal).
LIMIT = int(os.environ["ALETHEIA_LIMIT"]) if os.environ.get("ALETHEIA_LIMIT") else None

## Load the trained probe

`mean_probe.npz` carries the probe **direction** plus everything needed to
reproduce the exact feature extraction (`layer`, `model_id`, `max_seq_len`) and
scoring (the `mu`/`sigma`/`normalize`/`bias`/`score_std` standardization the probe
was trained with — apply it, don't project raw activations).

In [ ]:
CANDIDATES = ["submission/mean_probe.npz", "mean_probe.npz", "baselines/mean_probe.npz"]
WEIGHTS = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert WEIGHTS is not None, f"mean_probe.npz not found; looked in {CANDIDATES}"

z = np.load(WEIGHTS)
DIRECTION = z["direction"].astype(np.float64)
LAYER     = int(z["layer"])
MODEL_ID  = str(z["model_id"])
MAX_LEN   = int(z["max_seq_len"])
NORMALIZE = bool(z["normalize"]) if "normalize" in z.files else False
MU        = z["mu"].astype(np.float64)    if "mu" in z.files        else np.zeros_like(DIRECTION)
SIGMA     = z["sigma"].astype(np.float64) if "sigma" in z.files     else np.ones_like(DIRECTION)
BIAS      = float(z["bias"])              if "bias" in z.files      else 0.0
SCORE_STD = float(z["score_std"])         if "score_std" in z.files else 1.0
print(f"probe: model={MODEL_ID} | layer={LAYER} | dim={DIRECTION.shape[0]} | "
      f"max_len={MAX_LEN} | normalize={NORMALIZE}")

def probe_proba(X):
    """Pooled activations (N, hidden) -> deception probability in [0, 1], applying
    the probe's saved standardization (the sigmoid is a monotonic squash; AUROC is
    rank-based so it's preserved)."""
    X = np.asarray(X, dtype=np.float64)
    if NORMALIZE:
        X = (X - MU) / SIGMA
    s = (X @ DIRECTION + BIAS) / SCORE_STD
    return 1.0 / (1.0 + np.exp(-np.clip(s, -30.0, 30.0)))

## Build the model + tokenization

The probe is a fixed feature extractor trained on `google/gemma-3-27b-it`, so we
build that model for **every** dataset (no LoRA). `build_model` picks the right
nnsight wrapper; the weights live on NDIF (we only trace against a handle).

In [ ]:
model = build_model(MODEL_ID)
DECODER = decoder_layers(model)
tok = model.tokenizer
EOT = tok.convert_tokens_to_ids("<end_of_turn>")

def tokenize(messages):
    """Probe's tokenization: trim to the last assistant turn, merge a leading
    system turn into the first user turn, find the response span (minus trailing
    <end_of_turn>), and left-trim to MAX_LEN keeping the response. Returns
    (ids, start, end)."""
    msgs = list(messages)
    a = [i for i, m in enumerate(msgs) if m.get("role") == "assistant"]
    if a:
        msgs = msgs[: a[-1] + 1]
    norm, pending = [], None
    for m in msgs:
        if m["role"] == "system" and not norm:
            pending = m.get("content", ""); continue
        if not norm and m["role"] == "user" and pending is not None:
            norm.append({"role": "user", "content": pending + "\n\n" + m.get("content", "")})
            pending = None; continue
        norm.append(m)
    if not norm:
        norm = msgs
    ids, s, e = response_span(norm, tok)
    while e > s and ids[e - 1] == EOT:
        e -= 1
    if MAX_LEN and len(ids) > MAX_LEN:
        cut = min(len(ids) - MAX_LEN, s); ids, s, e = ids[cut:], s - cut, e - cut
        if len(ids) > MAX_LEN:
            c2 = len(ids) - MAX_LEN; ids, s, e = ids[c2:], max(0, s - c2), e - c2
    return ids, s, e

## Extract activations — one remote trace per batch

`batch_size * max_len` stays under the ~2560-token budget for the 27B model
(`8 * 300 = 2400`). Right-pad each batch; the attention mask hides the padding so
the response-span pooling is exact.

In [ ]:
BATCH_SIZE = 8
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

examples = load_examples(DATASET_NAME)
n = len(examples) if LIMIT is None else min(LIMIT, len(examples))
print(f"scoring {n}/{len(examples)} rows")

# Tokenize locally; rows with an empty response span fall back to 0.5.
toks, valid = [], []
for i in range(n):
    ids, s, e = tokenize(examples[i]["messages"])
    if e > s:
        toks.append((ids, s, e)); valid.append(i)

def extract(rows):
    out = []
    for b0 in range(0, len(rows), BATCH_SIZE):
        batch = rows[b0:b0 + BATCH_SIZE]
        w = max(len(ids) for ids, _, _ in batch)
        input_ids = torch.full((len(batch), w), PAD, dtype=torch.long)
        attn = torch.zeros((len(batch), w), dtype=torch.long)
        for j, (ids, _, _) in enumerate(batch):
            input_ids[j, :len(ids)] = torch.tensor(ids); attn[j, :len(ids)] = 1
        # One remote trace for this batch (the "regular" path: many traces).
        with model.trace({"input_ids": input_ids, "attention_mask": attn}, remote=True):
            h = DECODER[LAYER].output                                   # (B, T, hidden)
            pooled = torch.stack(
                [h[j, s:e].mean(0) for j, (_, s, e) in enumerate(batch)]).save()
        out.append(pooled.cpu().float().numpy())
        print(f"  batch {b0 // BATCH_SIZE + 1}/{(len(rows) + BATCH_SIZE - 1) // BATCH_SIZE}")
    return np.concatenate(out, axis=0)

scores = np.full(n, 0.5, dtype=float)
if valid:
    scores[np.array(valid)] = probe_proba(extract(toks))

## Write `submission.csv`

`write_submission` emits the grader's three columns (`index,deceptive,score`),
keyed on the dataset's `index`; `deceptive` defaults to `score >= 0.5`.

In [ ]:
write_submission(list(examples["index"])[:n], scores)